In [2]:
# ① 必要なライブラリ
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
import matplotlib.pyplot as plt
import gradio as gr

D:\Pytorch-\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [4]:
import sys
sys.executable

'D:\\Pytorch-\\.venv\\Scripts\\python.exe'

In [5]:
# ② ドライブをマウントして、重みファイルを読み込む
path = 'motorcycle_classifier.pth'

model = resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model.load_state_dict(torch.load(path, map_location=device))
model.to(device)
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [6]:
# ③ test_dsの代わりに、必要な情報だけを直接書く
classes = ['real', 'toy']  # ImageFolderが決めたclass_to_idxの順序
transform = ResNet18_Weights.DEFAULT.transforms()  # 学習時と同じ前処理

In [7]:
# ④ Grad-CAM用のフック
activations = {}
gradients = {}

def forward_hook(module, input, output):
    activations["value"] = output

def backward_hook(module, grad_input, grad_output):
    gradients["value"] = grad_output[0]

target_layer = model.layer4[-1]
target_layer.register_forward_hook(forward_hook)
target_layer.register_full_backward_hook(backward_hook)

In [8]:
def grad_cam(xb, class_idx):
    xb = xb.unsqueeze(0).to(device)
    xb.requires_grad_(True)

    model.zero_grad()
    logits = model(xb)
    score = logits[0, class_idx]
    score.backward()

    act = activations["value"][0]
    grad = gradients["value"][0]
    weights_c = grad.mean(dim=(1, 2))
    cam = torch.zeros(act.shape[1:], device=act.device)
    for c in range(act.shape[0]):
        cam += weights_c[c] * act[c]
    cam = torch.relu(cam)
    cam = cam - cam.min()
    cam = cam / (cam.max() + 1e-8)
    cam = F.interpolate(cam.unsqueeze(0).unsqueeze(0), size=(224, 224), mode="bilinear", align_corners=False)
    return cam.squeeze().detach().cpu().numpy()

# ⑤ 予測関数(test_dsへの参照なし)
mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])

def predict_with_gradcam(img: Image.Image):
    xb = transform(img.convert("RGB"))

    with torch.inference_mode():
        probs = torch.softmax(model(xb.unsqueeze(0).to(device)), dim=1)[0]
    pred = probs.argmax().item()

    cam = grad_cam(xb, class_idx=pred)
    base_img = (xb * std[:, None, None] + mean[:, None, None]).permute(1, 2, 0).clamp(0, 1).numpy()

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(base_img)
    ax.imshow(cam, cmap="jet", alpha=0.5)
    ax.axis("off")

    label_dict = {classes[i]: float(probs[i]) for i in range(len(classes))}
    return label_dict, fig

# ⑥ Gradio起動
demo = gr.Interface(
    fn=predict_with_gradcam,
    inputs=gr.Image(type="pil"),
    outputs=[gr.Label(num_top_classes=2), gr.Plot(label="判断の根拠(Grad-CAM)")],
    title="toy / real モーターサイクル判定 + 判断根拠の可視化",
)
demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.
